In [ ]:
# Load packages
import numpy as np 
import pandas as pd 
import statsmodels.api as sm
from linearmodels.iv.model import IV2SLS

# For details, review the documenation:
#   - numpy: https://numpy.org/doc/stable/ 
#   - pandas: https://pandas.pydata.org/docs/user_guide/index.html
#   - statsmodels: https://www.statsmodels.org/stable/index.html
#  - linearmodels: https://pypi.org/project/linearmodels/

### 1. Redoing Card's IV Example - With Packages

In the first introductory example we will redo the IV example from the Card 1993 paper. We will do it in two ways, first using a package from statsmodels and then doing it manually in section 2. For details on the problem look at Exercise 7.

In [ ]:
# First load data
card = pd.read_stata("card.dta")

In [ ]:
# Stata equivalent:
#   We use the statsmodels package (loaded above)
#   regress lwage educ exper expersq black south married smsa smsa66 reg662-reg669

# OLS
ols_reg = sm.OLS.from_formula("lwage ~ educ + exper + expersq + black + south + married + smsa + smsa66 + reg662 + reg663 + reg664 + reg665 + reg666 + reg667 + reg668 + reg669", 
              data = card).fit()
ols_reg.summary()

In [ ]:
# A more pythonic way to do the same thing. Define a list of independent variables
x_vars = [
    "educ",
    "exper",
    "expersq",
    "black",
    "south",
    "married",
    "smsa",
    "smsa66",
    "reg662",
    "reg663",
    "reg664",
    "reg665",
    "reg666",
    "reg667",
    "reg668",
    "reg669"
]
# Generate X container
X = sm.add_constant(card[x_vars])
X = X.rename(columns={"const": "intercept"})
# Ensure there are no NaNs
X = X.dropna()
ols_reg = sm.OLS(
    endog=card.loc[X.index, "lwage"],
    exog=X
)
ols_reg = ols_reg.fit()
ols_reg.summary()


In [ ]:
# NOTE on standard errors: linearmodels.IV2SLS defaults to ROBUST
# (heteroskedasticity-consistent) standard errors, whereas the manual
# implementations below use classical (homoskedastic) standard errors.
# Point estimates are identical; the SEs will differ for this reason.
# To reproduce the classical SEs from the package, pass cov_type="unadjusted".
#
# Stata equivalent (robust):
#   ivregress 2sls lwage exper expersq black south married smsa smsa66 \
#       reg662-reg669 (educ = nearc4), vce(robust)

# Load IV from linearmodels and do reduced analysis (already loaded above)
#from linearmodels.iv.model import IV2SLS

# Generate iv formula. First x_vars without educ
x_vars_no_educ = [
    "exper",
    "expersq",
    "black",
    "south",
    "married",
    "smsa",
    "smsa66",
    "reg662",
    "reg663",
    "reg664",
    "reg665",
    "reg666",
    "reg667",
    "reg668",
    "reg669"
]
# Generate the formula for the IV regression
# Note that we use the formula syntax from linearmodels
iv_formula = "lwage ~ 1 + " + " + ".join(x_vars_no_educ) + " + [educ ~ nearc4]"

iv_reg = IV2SLS.from_formula(iv_formula, card).fit()
iv_reg.summary

#### First stage and weak-instrument check

Before trusting the IV estimate we check that the instrument `nearc4` is relevant, i.e. that it actually predicts the endogenous regressor `educ` in the first stage. The standard diagnostic is the first-stage F-statistic on the excluded instrument(s); a common rule of thumb is $F > 10$.


In [ ]:
# First-stage regression: educ on the instrument and all exogenous regressors
first_stage = sm.OLS.from_formula(
    "educ ~ nearc4 + " + " + ".join(x_vars_no_educ),
    data=card,
).fit()

# F-statistic for the excluded instrument (nearc4)
f_test = first_stage.f_test("nearc4 = 0")
print(f"First-stage coefficient on nearc4: {first_stage.params['nearc4']:.4f}")
print(f"First-stage F-statistic (nearc4):  {float(f_test.fvalue):.2f}")

# Stata equivalent:
#   regress educ nearc4 exper expersq black south married smsa smsa66 reg662-reg669
#   test nearc4


### 2. Redoing Card's IV Example - Manually

Recall the OLS formula:

$$\hat{\beta} = (X'X)^{-1}X'y$$

and for the standard errors:

$$\hat{\sigma}^2 = \frac{1}{n-k} \hat{u}'\hat{u}$$

$$\hat{Var}(\hat{\beta}) = \hat{\sigma}^2 (X'X)^{-1}$$

Note: you can find helpful derivation notes, e.g. [here](https://web.stanford.edu/~mrosenfe/soc_meth_proj3/matrix_OLS_NYU_notes.pdf).

In [ ]:
# First get the data
x = sm.add_constant(card[x_vars]).dropna()
# Get index of x
x_index = x.index
# Transform x to a matrix
x = x.values
# Get the dependent variable
y = card.loc[x_index, "lwage"]

In [ ]:
# Then calculate x'x
xpx = x.T @ x
# Use numpy for the inverse
xpx_inv = np.linalg.inv(xpx)
# Calculate the coefficients
beta = xpx_inv @ (x.T @ y)
beta

In [ ]:
# Check that those are the same as the OLS coefficients
np.allclose(ols_reg.params, beta)

#### Define function for OLS
Define re-usable function for OLS estimators. We will call this function in the 2SLS estimator below.

In [ ]:
def ols_formula(y, x):
    """
    Define a function for the OLS estimator.

    Args:
        y: dependent var, a 1d array/series
        x: independent vars, a 2d array (num_obs, num_xvars)
        
    Returns:
        coeffs: estimated OLS coefficients
        std_errors: estimated standard errors (assuming homosked.)
    """

    inverse_covars = np.linalg.inv(x.T @ x)
    # OLS estimator formula
    coeffs = inverse_covars @ (x.T @ y)

    # Now estimation of standard errors
    projection = x @ coeffs

    residuals = y - projection
    squared_sum_residuals = residuals @ residuals

    degrees_of_freedom = x.shape[0] - x.shape[1]
    covariance_est = (squared_sum_residuals / degrees_of_freedom) * inverse_covars
    std_errors = np.sqrt(np.diag(covariance_est))
    return coeffs, std_errors

In [ ]:
# Use formula and compare results
coeffs, std_errors = ols_formula(y, x)
np.allclose(ols_reg.params, coeffs)

### IV manually

Let $ y \in \mathbb{R}^n $ be the outcome variable, and let the regressor matrix be partitioned as
$$
X = \begin{bmatrix} X_o & x_e \end{bmatrix},
$$
where:

- $ X_o \in \mathbb{R}^{n \times k} $: matrix of exogenous regressors,
- $ x_e \in \mathbb{R}^{n \times 1} $: endogenous regressor,
- $ Z \in \mathbb{R}^{n \times \ell} $: matrix of instruments, including excluded instruments and the exogenous variables in $ X_o $.

---

#### Two-Stage Least Squares (2SLS) Estimator

**Stage 1 (First Stage):**  
Regress the endogenous regressor $ x_e $ on the instruments $ Z $:
$$
\hat{x}_e = P_Z x_e, \quad \text{where } P_Z = Z (Z^\top Z)^{-1} Z^\top
$$
is the projection matrix onto the column space of $ Z $.

**Stage 2 (Second Stage):**  
Regress $ y $ on the fitted regressor matrix:
$$
\hat{X} = \begin{bmatrix} X_o & \hat{x}_e \end{bmatrix}.
$$
Then the 2SLS estimator is given by:
$$
\hat{\beta}_{2SLS} = \left( \hat{X}^\top \hat{X} \right)^{-1} \hat{X}^\top y.
$$

---

In [ ]:
def two_sls_with_OLS(y, x_exog, x_to_instrument, z):
    """
    Two stage least squares estimator (estimated via two explicit OLS steps).

    Args:
        y: dependent variable, 1d array/series
        x_exog: exogenous regressors (everything EXCEPT the endogenous regressor)
        x_to_instrument: the endogenous regressor (instrumented in the first stage)
        z: instruments, 2d array (exogenous regressors + excluded instrument(s))

    Returns:
        coeffs: estimated coefficients (second stage)
        std_errors: classical (homosked.) standard errors    
    

    NOTE: The second-stage SEs are NOT the naive OLS SEs from regressing y on the
    fitted regressors. The variance must be computed using residuals based on the
    ORIGINAL endogenous regressor (x_to_instrument), not its first-stage fitted
    values. Using the fitted values would understate the residual variance and give
    wrong standard errors. 
    """
    # First stage: regress the endogenous regressor on the instruments z
    first_stage_ols_coeff, _ = ols_formula(x_to_instrument, z)
    # Get the fitted values from the first stage
    x_hat = z @ first_stage_ols_coeff
    
    # Second stage: regress y on [exogenous regressors, x_hat]
    x_total = np.hstack((x_exog, x_hat.reshape(-1, 1)))
    coeffs, _ = ols_formula(y, x_total)

    # Standard errors: rebuild the regressor matrix with the ORIGINAL endogenous
    # column and use the projection P_z to form (X' P_z X)^{-1}.
    x_orig = np.hstack((x_exog, x_to_instrument.reshape(-1, 1)))
    pz = z @ np.linalg.inv(z.T @ z) @ z.T
    bread = np.linalg.inv(x_orig.T @ pz @ x_orig)

    residuals = y - x_orig @ coeffs                 # residuals use ORIGINAL educ
    dof = x_orig.shape[0] - x_orig.shape[1]
    sigma2 = (residuals @ residuals) / dof
    cov = sigma2 * bread
    std_errors = np.sqrt(np.diag(cov))
    
    return coeffs, std_errors


In [ ]:
# Read out the instruments
z = sm.add_constant(card.loc[x_index][x_vars_no_educ + ["nearc4"]]).values
x_to_instrument = card.loc[x_index]["educ"].values
x_without_educ = sm.add_constant(card.loc[x_index][x_vars_no_educ]).values

# Calculate the 2SLS coefficients and (classical) standard errors
coeffs_2sls, se_2sls = two_sls_with_OLS(y, x_without_educ, x_to_instrument, z)

# Point estimates match the package exactly
print("Coefficients match package:", np.allclose(iv_reg.params, coeffs_2sls))

# The classical SEs match the package ONLY if the package also uses classical SEs.
# iv_reg above was fit with the default robust covariance, so they will differ.
#
# There is also a degrees-of-freedom convention to reconcile: our manual SEs divide
# the residual variance by (n - k), the small-sample correction that Stata's
# ivregress also uses. linearmodels divides by n unless debiased=True is set.
# We therefore refit with BOTH cov_type="unadjusted" (classical, not robust) and
# debiased=True (the n - k correction) to compare like-for-like:
iv_reg_classical = IV2SLS.from_formula(iv_formula, card).fit(
    cov_type="unadjusted", debiased=True
)
print("Classical SEs match package:",
      np.allclose(iv_reg_classical.std_errors, se_2sls))



#### Alternative Matrix Formulation

We can also express the 2SLS estimator using the projection matrix $ P_Z $ directly:
$$
\hat{\beta}_{2SLS} = \left( X^\top P_Z X \right)^{-1} X^\top P_Z y,
$$
where $ X = \begin{bmatrix} X_o & x_e \end{bmatrix} $ includes the endogenous regressor before projection. Here, $ P_Z $ again denotes the projection matrix:
$$
P_Z = Z (Z^\top Z)^{-1} Z^\top.
$$

---

In [ ]:
def two_sls_formula(y, x, z):
    """
    Matrix version of the two stage least squares estimator.

    y: dependent variable (n,)
    x: regressors INCLUDING the endogenous regressor (n, k)
    z: instruments = exogenous regressors + excluded instrument(s) (n, l)

    Returns the coefficients and classical (homoskedastic) standard errors.
    """
    pz = z @ np.linalg.inv(z.T @ z) @ z.T            # projection onto col(Z)
    bread = np.linalg.inv(x.T @ pz @ x)
    coeffs = bread @ (x.T @ pz @ y)

    # Classical 2SLS variance: sigma^2 * (X' P_z X)^{-1},
    # with residuals formed from the ORIGINAL X (not the projected X).
    residuals = y - x @ coeffs
    dof = x.shape[0] - x.shape[1]
    sigma2 = (residuals @ residuals) / dof
    cov = sigma2 * bread
    std_errors = np.sqrt(np.diag(cov))
    
    return coeffs, std_errors


In [ ]:
x_iv = sm.add_constant(card.loc[x_index][x_vars_no_educ + ["educ"]]).values
coeff_iv, se_iv = two_sls_formula(y, x_iv, z)

# Compare with the IV regression (point estimates, and classical SEs)
print("Coefficients match package:", np.allclose(iv_reg.params, coeff_iv))
print("Classical SEs match OLS-based 2SLS:", np.allclose(se_iv, se_2sls))


#### GMM Criterion Formulation of 2SLS

The 2SLS estimator can also be derived as a special case of the Generalized Method of Moments (GMM). The population moment condition is:
$$
\mathbb{E} \left[ Z^\top (y - X\beta) \right] = 0.
$$

The corresponding sample moment is:
$$
g_n(\beta) = \frac{1}{n} Z^\top (y - X\beta).
$$

The GMM criterion function is:
$$
Q_n(\beta) = g_n(\beta)^\top W g_n(\beta),
$$
where $ W $ is a symmetric positive definite weighting matrix. For 2SLS, the efficient choice (under homoskedasticity) is
$$
W = (Z^\top Z)^{-1},
$$
which yields $\hat{\beta}_{GMM} = (X^\top P_Z X)^{-1} X^\top P_Z y$, i.e. exactly the 2SLS estimator above.

Note, however, that our example is **just-identified**: there is one excluded instrument (`nearc4`) for one endogenous regressor (`educ`), so $\dim(Z) = \dim(X)$. In this case the sample moment condition $g_n(\beta) = 0$ can be solved exactly, the criterion attains $Q_n = 0$ at the solution, and the choice of $W$ is irrelevant — any positive definite $W$ gives the same $\hat{\beta}$. The weighting matrix only matters under **over-identification** (more excluded instruments than endogenous regressors). The code below therefore solves the moment condition directly by root-finding rather than minimizing $Q_n$.

In [ ]:
def gmm_moment_condition(beta, y, x, z):
    """
    GMM moment condition for 2SLS:
        g(beta) = Z' (y - X beta)
    The root of g(beta) gives the 2SLS estimate.
    
    Parameters:
        beta: (k,) array, guess for coefficients
        y: (n,) array, dependent variable
        x: (n, k) array, regressors (may include endogenous)
        z: (n, l) array, instruments = exogenous regressors + excluded instrument(s)

    Returns:
        moments: (l,) array, the sample moment vector
    """
    residuals = y - x @ beta
    moments = z.T @ residuals
    return moments

In [ ]:
from scipy.optimize import root

# Example usage
res = root(gmm_moment_condition, x0=np.zeros(x.shape[1]), args=(y, x_iv, z))

# Always verify the solver converged before trusting res.x
assert res.success, f"root finder did not converge: {res.message}"
res.success

In [ ]:
# Compare with the IV regression (point estimates only; the just-identified
# moment condition pins down beta but does not itself produce standard errors).
np.allclose(iv_reg.params, res.x)


### Interpretation

Comparing the two estimates of the return to schooling:

- **OLS:** the coefficient on `educ` is about **0.072** (≈ 7.2% per year of schooling).
- **IV (2SLS):** the coefficient on `educ` is about **0.120** (≈ 12% per year).

The IV estimate is *larger* than OLS. Two standard explanations:

1. **Measurement error / attenuation:** classical measurement error in `educ` biases OLS toward zero; IV corrects for it, pushing the estimate up.
2. **LATE interpretation:** with a binary instrument (proximity to a college, `nearc4`), 2SLS recovers a Local Average Treatment Effect — the return for *compliers*, individuals induced to acquire more schooling because they grew up near a college. These tend to be individuals from more disadvantaged backgrounds with high marginal returns to education, so the LATE can exceed the OLS average.

The first-stage F-statistic computed above (≈ 12) clears the conventional rule-of-thumb threshold of 10, so `nearc4` is a relevant — though not especially strong — instrument. The relatively modest F is itself worth noting: the wide IV confidence interval reflects this.
